# 🏦 Credit Risk Prediction Pipeline
This notebook builds a machine learning pipeline to predict loan approval status. We will walk through data cleaning, preprocessing, handling class imbalances, model training (Logistic Regression, Random Forest, Decision Trees, XGBoost), hyperparameter tuning, and advanced feature selection.

## 1. Import Libraries & Load Data
First, we import the necessary libraries for data manipulation and load our dataset.

In [2]:
# Importing the important libraries
import pandas as pd
import numpy as np

# Loading the dataset
cr = pd.read_csv(r"C:\Users\Asus\Downloads\CreditRisk.csv")

## 2. Data Cleaning & Missing Value Imputation
We handle missing values by filling categorical features with their mode (most frequent value) and continuous features with their mean.

In [3]:
# Data cleaning: Filling missing valuescr.Gender.fillna('Male', inplace = True)
cr.Married.fillna('Yes', inplace = True)
cr.Dependents.fillna(0, inplace = True)
cr.Self_Employed.fillna("No", inplace = True)
cr.LoanAmount.fillna(cr.LoanAmount.mean(), inplace = True)
cr.Loan_Amount_Term.fillna(cr.Loan_Amount_Term.mean(),inplace = True)
cr.Credit_History.fillna(0, inplace = True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_12372\3412435448.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cr.Married.fillna('Yes', inplace = True)
C:\Users\Asus\AppData\Local\Temp\ipykernel_12372\3412435448.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when

In [18]:
cr.isnull().sum()


Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [19]:
cr.select_dtypes(include ='object').columns

Index(['Loan_ID', 'Gender', 'Married', 'Education', 'Self_Employed',
       'Property_Area', 'Loan_Status'],
      dtype='object')

## 3. Data Preprocessing: Label Encoding
Machine learning models require numerical input. We will use `LabelEncoder` to convert our categorical text columns (like Gender, Married, etc.) into numeric labels.

In [4]:
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
le = LabelEncoder()

# Drop the Loan_ID column as it does not hold predictive value
cr = cr.drop(['Loan_ID'], axis = 1) 

# Apply LabelEncoder to all 'object' (categorical) columns
object_cols = cr.select_dtypes(include ='object').columns
cr[object_cols] = cr[object_cols].apply(le.fit_transform)

## 4. Train-Test Split
We split the data into training (80%) and testing (20%) sets to evaluate our model on unseen data. We use a `random_state` for reproducibility.

In [5]:
from sklearn.model_selection import train_test_split

# Splitting the dataset
cr_train, cr_test = train_test_split(cr, test_size =.2, random_state = 123) 

# Separating features (x) and target variable (y)
cr_train_x = cr_train.iloc[:,0:-1]
cr_train_y = cr_train.iloc[:,-1]
cr_test_x = cr_test.iloc[:,0:-1]
cr_test_y = cr_test.iloc[:,-1]

In [6]:
cr_train.Loan_Status.value_counts()

Loan_Status
1    578
0    206
Name: count, dtype: int64

## 5. Handling Class Imbalance
Imbalanced target classes can bias the model toward the majority class. We use **SMOTE (Synthetic Minority Over-sampling Technique)** to synthetically generate new examples for the minority class in the training set.

In [7]:
import imblearn
from imblearn.over_sampling import SMOTE

In [8]:
smt = SMOTE()
df1, df2 = smt.fit_resample(cr_train_x, cr_train_y)

In [9]:
df1.shape , df2.shape

((1156, 11), (1156,))

## 5. Handling Class Imbalance
Imbalanced target classes can bias the model toward the majority class. We use **SMOTE (Synthetic Minority Over-sampling Technique)** to synthetically generate new examples for the minority class in the training set.

In [10]:
# Cell 11
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df1_scaled = scaler.fit_transform(df1)  # ✓ Create new variable
cr_test_x_scaled = scaler.transform(cr_test_x)  # ✓ Create new variable

## 7. Model Building & Evaluation
We will train multiple base models to establish a performance baseline, evaluating them using a Confusion Matrix and Classification Report.

### 7.1 Logistic Regression & Decision Trees

In [11]:
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

# fit and predict
model.fit(df1_scaled, df2)

# Predict
pred = model.predict(cr_test_x_scaled)

# Evaluation - use cr_test_y (true labels), not cr_test_x (features)
tab = confusion_matrix(cr_test_y, pred)
print(tab)
print(classification_report(cr_test_y, pred))


[[ 37  26]
 [ 28 106]]
              precision    recall  f1-score   support

           0       0.57      0.59      0.58        63
           1       0.80      0.79      0.80       134

    accuracy                           0.73       197
   macro avg       0.69      0.69      0.69       197
weighted avg       0.73      0.73      0.73       197



In [36]:
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(criterion='entropy')

dt.fit(df1_scaled, df2)

# Predict
pred = dt.predict(cr_test_x_scaled)

# Evaluation - use cr_test_y (true labels), not cr_test_x (features)
tab = confusion_matrix(cr_test_y, pred)
print(tab)
print(classification_report(cr_test_y, pred))

[[ 34  29]
 [ 32 102]]
              precision    recall  f1-score   support

           0       0.52      0.54      0.53        63
           1       0.78      0.76      0.77       134

    accuracy                           0.69       197
   macro avg       0.65      0.65      0.65       197
weighted avg       0.69      0.69      0.69       197



In [35]:
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.tree import DecisionTreeClassifier
dt1 = DecisionTreeClassifier()


dt1.fit(df1_scaled, df2)

# Predict
pred = dt1.predict(cr_test_x_scaled)

# Evaluation - use cr_test_y (true labels), not cr_test_x (features)
tab = confusion_matrix(cr_test_y, pred)
print(tab)
print(classification_report(cr_test_y, pred))

[[ 31  32]
 [ 33 101]]
              precision    recall  f1-score   support

           0       0.48      0.49      0.49        63
           1       0.76      0.75      0.76       134

    accuracy                           0.67       197
   macro avg       0.62      0.62      0.62       197
weighted avg       0.67      0.67      0.67       197



### 7.2 Ensemble Models: Random Forest & XGBoost

In [37]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier()
rfc.fit(df1_scaled, df2)

# Predict
pred = rfc.predict(cr_test_x_scaled)

# Evaluation - use cr_test_y (true labels), not cr_test_x (features)
tab = confusion_matrix(cr_test_y, pred)
print(tab)
print(classification_report(cr_test_y, pred))

[[ 38  25]
 [ 20 114]]
              precision    recall  f1-score   support

           0       0.66      0.60      0.63        63
           1       0.82      0.85      0.84       134

    accuracy                           0.77       197
   macro avg       0.74      0.73      0.73       197
weighted avg       0.77      0.77      0.77       197



In [13]:
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report

# Quick XGBoost
xgb_model = xgb.XGBClassifier(random_state=123)
xgb_model.fit(df1_scaled, df2)
pred_xgb = xgb_model.predict(cr_test_x_scaled)

print(confusion_matrix(cr_test_y, pred_xgb))
print(classification_report(cr_test_y, pred_xgb))

[[ 37  26]
 [ 20 114]]
              precision    recall  f1-score   support

           0       0.65      0.59      0.62        63
           1       0.81      0.85      0.83       134

    accuracy                           0.77       197
   macro avg       0.73      0.72      0.72       197
weighted avg       0.76      0.77      0.76       197



In [43]:
# First install XGBoost if not already installed
# !pip install xgboost

import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

# Define the parameter grid
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 3, 5]
}

# Create XGBoost model
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=123,
    eval_metric='logloss'
)

# GridSearchCV
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,  # Use all processors
    verbose=2
)

# Fit the model
grid_search_xgb.fit(df1_scaled, df2)

# Best parameters and score
print("Best Parameters:", grid_search_xgb.best_params_)
print("Best Cross-Validation Score:", grid_search_xgb.best_score_)

# Predict with best model
best_xgb = grid_search_xgb.best_estimator_
pred_xgb = best_xgb.predict(cr_test_x_scaled)

# Evaluation
tab_xgb = confusion_matrix(cr_test_y, pred_xgb)
print("\nConfusion Matrix:")
print(tab_xgb)
print("\nClassification Report:")
print(classification_report(cr_test_y, pred_xgb))

Fitting 5 folds for each of 2916 candidates, totalling 14580 fits
Best Parameters: {'colsample_bytree': 0.7, 'gamma': 0.2, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.7}
Best Cross-Validation Score: 0.8686221824152858

Confusion Matrix:
[[ 31  32]
 [ 19 115]]

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.49      0.55        63
           1       0.78      0.86      0.82       134

    accuracy                           0.74       197
   macro avg       0.70      0.68      0.68       197
weighted avg       0.73      0.74      0.73       197



## 8. Hyperparameter Tuning
To squeeze out the best performance from our strongest model (XGBoost), we tune its hyperparameters using Cross-Validation techniques.

### 8.1 GridSearchCV

In [51]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

param_dist_xgb = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 6, 7],  # Default is 6
    'learning_rate': [0.01, 0.05, 0.1, 0.3,0.0001],  # Default is 0.3
    'subsample': [0.7, 0.8, 1.0],  # Default is 1.0
    'colsample_bytree': [0.7, 0.8, 1.0],  # Default is 1.0
    'gamma': [0, 0.1, 0.2],  # Default is 0
    'min_child_weight': [1, 3, 5]  # Default is 1
}

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=123,
    eval_metric='logloss'
)

random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=50,  # Try 50 random combinations
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=123,
    verbose=2
)

random_search.fit(df1_scaled, df2)
print("Best Parameters:", random_search.best_params_)
pred_xgb = random_search.best_estimator_.predict(cr_test_x_scaled)
print(confusion_matrix(cr_test_y, pred_xgb))
print(classification_report(cr_test_y, pred_xgb))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best Parameters: {'subsample': 0.7, 'n_estimators': 50, 'min_child_weight': 3, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.7}
[[ 37  26]
 [ 24 110]]
              precision    recall  f1-score   support

           0       0.61      0.59      0.60        63
           1       0.81      0.82      0.81       134

    accuracy                           0.75       197
   macro avg       0.71      0.70      0.71       197
weighted avg       0.74      0.75      0.75       197



In [46]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.metrics import confusion_matrix, classification_report, make_scorer, f1_score
import numpy as np

# Quick XGBoost
xgb_model = xgb.XGBClassifier(random_state=123)

# Cross-validation with multiple metrics
cv_scores = cross_validate(
    xgb_model, 
    df1_scaled, 
    df2, 
    cv=5,  # 5-fold cross-validation
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print("="*60)
print("CROSS-VALIDATION RESULTS (5-Fold)")
print("="*60)
print(f"Accuracy:  {cv_scores['test_accuracy'].mean():.4f} (+/- {cv_scores['test_accuracy'].std():.4f})")
print(f"Precision: {cv_scores['test_precision'].mean():.4f} (+/- {cv_scores['test_precision'].std():.4f})")
print(f"Recall:    {cv_scores['test_recall'].mean():.4f} (+/- {cv_scores['test_recall'].std():.4f})")
print(f"F1-Score:  {cv_scores['test_f1'].mean():.4f} (+/- {cv_scores['test_f1'].std():.4f})")
print("="*60)

# Train on full dataset and predict
xgb_model.fit(df1_scaled, df2)
pred_xgb = xgb_model.predict(cr_test_x_scaled)

print("\nTEST SET RESULTS")
print("="*60)
print("Confusion Matrix:")
print(confusion_matrix(cr_test_y, pred_xgb))
print("\nClassification Report:")
print(classification_report(cr_test_y, pred_xgb))

CROSS-VALIDATION RESULTS (5-Fold)
Accuracy:  0.8479 (+/- 0.0852)
Precision: 0.8611 (+/- 0.1313)
Recall:    0.8615 (+/- 0.0380)
F1-Score:  0.8553 (+/- 0.0695)

TEST SET RESULTS
Confusion Matrix:
[[ 39  24]
 [ 20 114]]

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.62      0.64        63
           1       0.83      0.85      0.84       134

    accuracy                           0.78       197
   macro avg       0.74      0.73      0.74       197
weighted avg       0.77      0.78      0.77       197



## 9. Feature Selection
Not all features contribute equally to the model's predictive power. Removing noisy features can improve model generalization.

### 9.1 Boruta
Boruta works as a wrapper algorithm around Random Forest/XGBoost to capture all important, interesting features.

In [57]:
from boruta import BorutaPy
import xgboost as xgb
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# XGBoost estimator
xgb_estimator = xgb.XGBClassifier(n_estimators=100, random_state=123, n_jobs=-1)

# Boruta selector
boruta_selector = BorutaPy(
    estimator=xgb_estimator,
    n_estimators='auto',
    max_iter=100,
    random_state=123,
    verbose=2
)

# Fit Boruta
boruta_selector.fit(df1_scaled, df2)

# Create feature ranking DataFrame
feature_ranks = pd.DataFrame({
    'Feature': cr_train_x.columns,
    'Ranking': boruta_selector.ranking_,
    'Selected': boruta_selector.support_,
    'Tentative': boruta_selector.support_weak_
})
feature_ranks = feature_ranks.sort_values('Ranking')

print("\n" + "="*60)
print("FEATURE RANKINGS")
print("="*60)
print(feature_ranks)
print("="*60)

# Selected features
selected_features = feature_ranks[feature_ranks['Selected'] == True]['Feature'].values
print(f"\nConfirmed important features: {list(selected_features)}")

# Transform data
df1_boruta = df1_scaled[:, boruta_selector.support_]
cr_test_x_boruta = cr_test_x_scaled[:, boruta_selector.support_]

# Train and evaluate
xgb_model = xgb.XGBClassifier(random_state=123)
xgb_model.fit(df1_boruta, df2)
pred_xgb = xgb_model.predict(cr_test_x_boruta)

print("\nResults with Boruta-selected features:")
print(confusion_matrix(cr_test_y, pred_xgb))
print(classification_report(cr_test_y, pred_xgb))

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	11
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	2
Tentative: 	8
Rejected: 	1
Iteration: 	9 / 100
Confirmed: 	2
Tentative: 	8
Rejected: 	1
Iteration: 	10 / 100
Confirmed: 	2
Tentative: 	8
Rejected: 	1
Iteration: 	11 / 100
Confirmed: 	2
Tentative: 	8
Rejected: 	1
Iteration: 	12 / 100
Confirmed: 	2
Tentative: 	5
Rejected: 	4
Iteration: 	13 / 100
Confirmed: 	2
Tentative: 	5
Rejected: 	4
Iteration: 	14 / 100
Confirmed: 	2
Tentative: 	5
Rejected: 	4
Iteration: 	15 / 100
Confirmed: 	2
Tentative: 	5
Rejected: 	4
Iteration: 	16 / 100
Confirmed: 	2
Tentative: 	5
Rejected: 	4
Iteration:

### 9.2 Recursive Feature Elimination (RFE)
RFE recursively removes the weakest features until the specified number of features is reached. Here, we extract the top 8 features.

In [15]:
from sklearn.feature_selection import RFE

# Initialize RFE with XGBoost to select the top 8 features
xgb_estimator = xgb.XGBClassifier(random_state=123, n_estimators=50)
rfe = RFE(estimator=xgb_estimator, n_features_to_select=8, step=1)
rfe.fit(df1_scaled, df2)

# Display the final chosen features
selected_features_list = cr_train_x.columns[rfe.support_].tolist()
print(f"Selected features: {selected_features_list}")

# Transform the datasets to only include the top 8 features
df1_selected = df1_scaled[:, rfe.support_]
cr_test_x_selected = cr_test_x_scaled[:, rfe.support_]

# Train final XGBoost model on selected features
xgb_final = xgb.XGBClassifier(random_state=123)
xgb_final.fit(df1_selected, df2)
pred_final = xgb_final.predict(cr_test_x_selected)

print("\nFinal Model Results (Top 8 Features):")
print(confusion_matrix(cr_test_y, pred_final))
print(classification_report(cr_test_y, pred_final))

Selected features: ['Married', 'Dependents', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']

Final Model Results (Top 8 Features):
[[ 38  25]
 [ 22 112]]
              precision    recall  f1-score   support

           0       0.63      0.60      0.62        63
           1       0.82      0.84      0.83       134

    accuracy                           0.76       197
   macro avg       0.73      0.72      0.72       197
weighted avg       0.76      0.76      0.76       197

